In [ ]:
import pandas as pd

cofs = pd.read_csv('../../../data/predictor/cof_matched/data_substrate_cofs.csv')['smiles'].tolist()
cofs

In [ ]:
from rdkit import Chem

endings = [
    'O=*',
    'C#N',
    'Cl-*',
    'Br-*',
    '[NH2]',
    '[HO]',
]

def check_multiple_occurrences(mol, smarts_list):
    """
    Checks if a molecule contains at least two occurrences of any
    single SMARTS pattern from a provided list.

    Args:
        mol (rdkit.Chem.Mol): The RDKit molecule object.
        smarts_list (list of str): A list of SMARTS strings to check.

    Returns:
        list of str: A list of SMARTS patterns that have at least
                     two occurrences in the molecule.
    """
    if mol is None:
        return []

    matched_patterns = set()
    i = 0
    for pattern_str in smarts_list:
        pattern = Chem.MolFromSmarts(pattern_str)
        if pattern is None:
            print(f"Warning: Invalid SMARTS pattern ignored: {pattern_str}")
            continue
        matches = mol.GetSubstructMatches(pattern)
        i += len(matches)
        if len(matches) >= 2:
            matched_patterns.add(pattern_str)

    return i >= 2

In [ ]:
import numpy as np

dict_count = {
    k: 0 for k in endings
}
counts = []

for smi in cofs:
    mol = Chem.MolFromSmiles(smi)
    matched = check_multiple_occurrences(mol, endings)
    counts.append(len(matched))
    for m in matched:
        dict_count[m] += 1

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
import itertools
from rdkit.Chem import rdmolops, rdDepictor, GetDistanceMatrix

def get_diameter_path(mol: Chem.Mol):
    # Calculate Distance Matrix
    dm = GetDistanceMatrix(mol)

    # Get diameter value
    diameter_val = int(np.max(dm))

    # Find start and end atoms (indices of the max value in the matrix)
    flat_idx = np.argmax(dm)
    source_idx, target_idx = np.unravel_index(flat_idx, dm.shape)

    # Get the path atoms
    path_atoms = list(rdmolops.GetShortestPath(mol, int(source_idx), int(target_idx)))

    # Get the path bonds (for nice continuous lines)
    path_bonds = []
    for i in range(len(path_atoms) - 1):
        bond = mol.GetBondBetweenAtoms(path_atoms[i], path_atoms[i+1])
        if bond:
            path_bonds.append(bond.GetIdx())

    return diameter_val, path_atoms, path_bonds


def visualize_molecule_diameter(mol, smarts_list):
    """
    Finds the maximum distance between atoms of the same pattern,
    highlights the path, and returns an SVG image.
    """
    if mol is None: return None

    # 1. Setup molecule for drawing
    mol = Chem.Mol(mol) # Make a copy
    rdDepictor.Compute2DCoords(mol)
    dist_matrix = rdmolops.GetDistanceMatrix(mol)

    max_dist = -1
    best_pair = (None, None)

    for pattern_str in smarts_list:
        pattern = Chem.MolFromSmarts(pattern_str)
        if not pattern: continue
        matches = mol.GetSubstructMatches(pattern)
        if len(matches) < 2: continue

        for m1, m2 in itertools.combinations(matches, 2):
            heavy1 = [i for i in m1 if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
            heavy2 = [i for i in m2 if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]

            for i in heavy1:
                for j in heavy2:
                    d = dist_matrix[i][j]
                    if d > max_dist:
                        max_dist = d
                        best_pair = (i, j)

    if best_pair[0] is None:
        print("No patterns matched twice.")
        return None

    atom_path = rdmolops.GetShortestPath(mol, best_pair[0], best_pair[1])

    bond_path = []
    for i in range(len(atom_path) - 1):
        bond = mol.GetBondBetweenAtoms(atom_path[i], atom_path[i+1])
        bond_path.append(bond.GetIdx())

    drawer = rdMolDraw2D.MolDraw2DSVG(500, 500)

    # Custom highlighting colors (RGB)
    highlight_atom_colors = {idx: (0.204, 1, 0.941, 0.2) for idx in atom_path}
    highlight_atom_colors[best_pair[0]] = (0.204, 1, 0.941, 0.2) # Start
    highlight_atom_colors[best_pair[1]] = (0.204, 1, 0.941, 0.2) # End

    highlight_bond_colors = {idx: (0.204, 1, 0.941, 0.4) for idx in bond_path}

    # Draw
    drawer.DrawMolecule(
        mol,
        highlightAtoms=atom_path,
        highlightAtomColors=highlight_atom_colors,
        highlightBonds=bond_path,
        highlightBondColors=highlight_bond_colors,
        legend=f'End-group diameter: {max_dist}'
    )
    drawer.FinishDrawing()

    print(f"Highlighted Diameter: {int(max_dist)} bonds.")
    return drawer.GetDrawingText()

def save_diameter_svg(mol, filename, width=500, height=500):
    d_val, atoms, bonds = get_diameter_path(mol)

    # Initialize the SVG drawer
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)

    opts = drawer.drawOptions()
    opts.addStereoAnnotation = False

    highlight_atom_colors = {idx: (0.204, 1, 0.941, 0.2) for idx in atoms}
    highlight_bond_colors = {idx: (0.204, 1, 0.941, 0.4) for idx in bonds}

    # Draw the molecule
    drawer.DrawMolecule(
        mol,
        highlightAtoms=atoms,
        highlightBonds=bonds,
        highlightAtomColors=highlight_atom_colors,
        highlightBondColors=highlight_bond_colors,
        legend=f"Diameter: {d_val}"
    )

    # Finalize and get the SVG string
    drawer.FinishDrawing()
    svg_string = drawer.GetDrawingText()
    return svg_string

    # Write to file
    # with open(filename, 'w') as f:
    #     f.write(svg_string)
    # print(f"Saved: {filename}")

def filtered_diameter(smiles):
    """
    Calculates the maximum distance (diameter) between occurrences of the
    same SMARTS pattern in a molecule, considering only heavy atoms.

    Args:
        mol (rdkit.Chem.Mol): The RDKit molecule.
        smarts_list (list of str): List of SMARTS strings to analyze.

    Returns:
        int: The maximum path length found, or 0 if no pattern appears twice.
    """
    patterns = [
        'O=*',
        'C#N',
        'Cl-*',
        'Br-*',
        '[NH2]',
        '[HO]',
    ]
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    dist_matrix = rdmolops.GetDistanceMatrix(mol)
    max_diameter = 0
    for pattern_str in patterns:
        pattern = Chem.MolFromSmarts(pattern_str)
        if pattern is None:
            continue

        # Get all matches (each match is a tuple of atom indices)
        matches = mol.GetSubstructMatches(pattern)

        # We need at least two occurrences to have a "distance between them"
        if len(matches) < 2:
            continue

        for match_idx_a, match_idx_b in itertools.combinations(range(len(matches)), 2):
            match_a = matches[match_idx_a]
            match_b = matches[match_idx_b]

            # Filter matches to include only non-hydrogen atom indices
            heavy_atoms_a = [i for i in match_a if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
            heavy_atoms_b = [i for i in match_b if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]

            # Check the distance between every heavy atom in A and every heavy atom in B
            for atom_i in heavy_atoms_a:
                for atom_j in heavy_atoms_b:
                    dist = dist_matrix[atom_i][atom_j]
                    if dist > max_diameter:
                        max_diameter = int(dist)

    return max_diameter

smi = cofs[19]
# for i, smi in enumerate(cofs[:20]):
#     print(i)
img_f = visualize_molecule_diameter(
    Chem.MolFromSmiles(smi),
    endings
)
img_d = save_diameter_svg(
    Chem.MolFromSmiles(smi),
    ''
)

In [ ]:
import re

def hex_replace(match):
    full_style = match.group(0)
    # Find the hex code inside the style string
    hex8 = re.search(r'#([0-9A-Fa-f]{6})([0-9A-Fa-f]{2})', full_style)
    if hex8:
        color, alpha = hex8.groups()
        opacity = round(int(alpha, 16) / 255.0, 3)
        # Replace 8-digit with 6-digit
        fixed = full_style.replace(f"#{color}{alpha}", f"#{color}")
        # Add opacity attributes before the closing quote
        return fixed[:-1] + f";fill-opacity:{opacity};stroke-opacity:{opacity}'"
    return full_style

img_f = re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, img_f)
img_d = re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, img_d)

with open(f'diameter.svg', 'w') as f:
    f.write(img_d)
with open('filtered_diameter.svg', 'w') as f:
    f.write(img_f)

In [ ]:
import matplotlib.pyplot as plt
labels = list(dict_count.keys())
values = list(dict_count.values())
plt.bar(labels, values)
plt.xlabel('Functional Group SMARTS')
plt.ylabel('Number of COFs with Even Occurrences')
plt.title('COFs with Even Occurrences of Functional Groups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(counts, bins=np.arange(0, max(counts)+2)-0.5, edgecolor='black')
plt.xlabel('Number of Different Functional Groups with Even Occurrences')
plt.ylabel('Number of COFs')
plt.title('Distribution of COFs by Number of Functional Groups with Even Occurrences')
plt.xticks(range(0, max(counts)+1))
plt.tight_layout()
plt.show()